In [3]:
!pip install -q open_clip_torch faiss-cpu pillow tqdm pandas pytesseract transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00


In [4]:
# =========================================================
# HIGH QUALITY HYBRID IMAGE RETRIEVAL PIPELINE
# CPU/GPU FRIENDLY
# FIXED + OPTIMIZED VERSION
# =========================================================

# =========================================================
# INSTALLS
# =========================================================

# !pip install -q sentence-transformers faiss-cpu pillow pytesseract tqdm

# =========================================================
# IMPORTS
# =========================================================

import os
import json
import time
import faiss
import torch
import numpy as np

from PIL import Image
from tqdm import tqdm
from collections import Counter
from sentence_transformers import SentenceTransformer
from IPython.display import display, Image as IPyImage

import pytesseract

# =========================================================
# CONFIG
# =========================================================

IMAGE_FOLDER = "/kaggle/input/datasets/chknaren/images-test"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TOP_K = 10

# =========================================================
# LOAD MODELS
# =========================================================

print("Loading CLIP Model...")

visual_embedder = SentenceTransformer(
    "clip-ViT-B-32",
    device=DEVICE
)

print("Loading Text Model...")

text_embedder = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=DEVICE
)

# =========================================================
# EMBEDDING FUNCTIONS
# =========================================================

def get_visual_embedding(image):

    emb = visual_embedder.encode(
        image,
        normalize_embeddings=True
    )

    return np.array(emb).astype("float32")


def get_clip_text_embedding(text):

    emb = visual_embedder.encode(
        text,
        normalize_embeddings=True
    )

    return np.array(emb).astype("float32")


def get_text_embedding(text):

    emb = text_embedder.encode(
        text,
        normalize_embeddings=True
    )

    return np.array(emb).astype("float32")

# =========================================================
# HELPERS
# =========================================================

def load_image(path):

    image = Image.open(path).convert("RGB")

    image.thumbnail((512, 512))

    return image


def extract_dominant_colors(image, k=5):

    img = image.resize((64, 64))

    pixels = np.array(img).reshape(-1, 3)

    pixels = (pixels // 32) * 32

    colors = Counter(map(tuple, pixels))

    top = colors.most_common(k)

    return [[int(v) for v in c] for c, _ in top]


def extract_ocr(image):

    try:

        gray = image.convert("L")

        text = pytesseract.image_to_string(
            gray,
            config="--psm 6"
        )

        return text.strip()

    except:

        return ""

# =========================================================
# QUERY TYPE DETECTOR
# =========================================================

def is_text_query(query):

    keywords = [
        "equation",
        "formula",
        "code",
        "error",
        "screenshot",
        "text",
        "message",
        "notes",
        "document",
        "pdf",
        "article",
        "linear algebra"
    ]

    query = query.lower()

    return any(k in query for k in keywords)

# =========================================================
# LOAD IMAGE PATHS
# =========================================================

image_paths = []

for root, _, files in os.walk(IMAGE_FOLDER):

    for file in files:

        if file.lower().endswith(
            (".jpg", ".jpeg", ".png", ".webp")
        ):

            image_paths.append(
                os.path.join(root, file)
            )

print(f"\nTOTAL IMAGES: {len(image_paths)}")

# =========================================================
# STORAGE
# =========================================================

metadata_store = []

visual_embeddings = []

text_embeddings = []

# =========================================================
# INGESTION
# =========================================================

start = time.time()

for path in tqdm(image_paths):

    try:

        image = load_image(path)

        # =====================================
        # VISUAL EMBEDDING
        # =====================================

        vis_emb = get_visual_embedding(image)

        # =====================================
        # OCR
        # =====================================

        ocr_text = extract_ocr(image)

        # =====================================
        # BETTER TEXT REPRESENTATION
        # =====================================

        filename = os.path.basename(path)

        combined_text = f"""
        Filename: {filename}
        OCR: {ocr_text}
        """

        txt_emb = get_text_embedding(combined_text)

        # =====================================
        # COLORS
        # =====================================

        colors = extract_dominant_colors(image)

        # =====================================
        # STORE
        # =====================================

        metadata_store.append({

            "path": path,

            "filename": filename,

            "ocr_text": ocr_text,

            "combined_text": combined_text,

            "colors": colors,

            "has_text": len(ocr_text) > 10
        })

        visual_embeddings.append(vis_emb)

        text_embeddings.append(txt_emb)

    except Exception as e:

        print(f"\nFAILED: {path}")

        print(e)

# =========================================================
# INGESTION COMPLETE
# =========================================================

elapsed = time.time() - start

print(f"\n✅ Ingestion completed in {elapsed:.2f} sec")

# =========================================================
# BUILD VISUAL INDEX
# =========================================================

visual_matrix = np.array(
    visual_embeddings
).astype("float32")

visual_index = faiss.IndexFlatIP(
    visual_matrix.shape[1]
)

visual_index.add(visual_matrix)

# =========================================================
# BUILD TEXT INDEX
# =========================================================

text_matrix = np.array(
    text_embeddings
).astype("float32")

text_index = faiss.IndexFlatIP(
    text_matrix.shape[1]
)

text_index.add(text_matrix)

# =========================================================
# SAVE METADATA
# =========================================================

with open("metadata.json", "w") as f:

    json.dump(metadata_store, f, indent=2)

print(
    f"\nVisual Index: {visual_index.ntotal}"
)

print(
    f"Text Index: {text_index.ntotal}"
)

# =========================================================
# HYBRID SEARCH
# =========================================================

def hybrid_search(query, top_k=8):

    start_time = time.time()

    text_query = is_text_query(query)

    # =====================================================
    # CLIP VISUAL SEARCH
    # IMPORTANT FIX:
    # USE CLIP TEXT ENCODER
    # =====================================================

    q_vis = get_clip_text_embedding(query)

    _, vis_idx = visual_index.search(
        np.array([q_vis]).astype("float32"),
        top_k * 5
    )

    # =====================================================
    # TEXT SEARCH
    # =====================================================

    q_txt = get_text_embedding(query)

    _, txt_idx = text_index.search(
        np.array([q_txt]).astype("float32"),
        top_k * 5
    )

    # =====================================================
    # SCORE FUSION
    # =====================================================

    scores = {}

    # ==========================================
    # QUERY ADAPTIVE WEIGHTING
    # ==========================================

    if text_query:

        vis_weight = 0.25
        txt_weight = 0.75

    else:

        vis_weight = 0.80
        txt_weight = 0.20

    # ==========================================
    # VISUAL RESULTS
    # ==========================================

    for rank, idx in enumerate(vis_idx[0]):

        score = vis_weight * (
            1.0 / (rank + 1)
        )

        scores[idx] = scores.get(idx, 0) + score

    # ==========================================
    # TEXT RESULTS
    # ==========================================

    for rank, idx in enumerate(txt_idx[0]):

        score = txt_weight * (
            1.0 / (rank + 1)
        )

        scores[idx] = scores.get(idx, 0) + score

    # =====================================================
    # FINAL SORT
    # =====================================================

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    # =====================================================
    # FORMAT RESULTS
    # =====================================================

    results = []

    for idx, score in ranked:

        meta = metadata_store[idx]

        results.append({

            "score": float(score),

            "path": meta["path"],

            "filename": meta["filename"],

            "ocr_snippet": meta[
                "ocr_text"
            ][:200],

            "has_text": meta["has_text"],

            "colors": meta["colors"]
        })

    elapsed = (
        time.time() - start_time
    ) * 1000

    print(
        f"\nQuery Time: {elapsed:.2f} ms"
    )

    print(
        f"Text Query: {text_query}"
    )

    return results

# =========================================================
# DISPLAY RESULTS
# =========================================================

def show_search_results(query, top_k=8):

    results = hybrid_search(
        query,
        top_k=top_k
    )

    print(f"\n🔍 QUERY: {query}\n")

    for rank, res in enumerate(results, 1):

        print("=" * 80)

        print(
            f"RANK #{rank}"
        )

        print(
            f"SCORE: {res['score']:.4f}"
        )

        print(
            f"FILE: {res['filename']}"
        )

        print(
            f"COLORS: {res['colors']}"
        )

        if res["has_text"]:

            print(
                f"OCR: {res['ocr_snippet']}"
            )

        display(
            IPyImage(
                filename=res["path"],
                width=480
            )
        )

# =========================================================
# TESTS
# =========================================================
'''
show_search_results(
    "mountain sunset",
    top_k=6
)

show_search_results(
    "beach with water ocean",
    top_k=6
)

show_search_results(
    "person in red shirt",
    top_k=6
)

show_search_results(
    "linear algebra notes",
    top_k=6
)

show_search_results(
    "error message screenshot",
    top_k=6
)
'''

Loading CLIP Model...


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


TOTAL IMAGES: 379


100%|██████████| 379/379 [04:01<00:00,  1.57it/s]


✅ Ingestion completed in 241.79 sec

Visual Index: 379
Text Index: 379


'\nshow_search_results(\n    "mountain sunset",\n    top_k=6\n)\n\nshow_search_results(\n    "beach with water ocean",\n    top_k=6\n)\n\nshow_search_results(\n    "person in red shirt",\n    top_k=6\n)\n\nshow_search_results(\n    "linear algebra notes",\n    top_k=6\n)\n\nshow_search_results(\n    "error message screenshot",\n    top_k=6\n)\n'